<a href="https://colab.research.google.com/github/Sprg72/Data-Engineer/blob/main/notebooks/pyspark_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# step1
!pip install findspark pyspark

In [2]:
# step2
import findspark
findspark.init()

In [3]:
#step3
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('myapp').getOrCreate()

In [9]:
sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=myapp>



```
# input file : emp1.txt

101, amar,90000,m,11
102, amala,20000,f,12
103, ankit,40000,m,13
104, ankita,60000,f,13
105, anusha,110000,f,12
106, anuz,20000,m,11
107, akash,100000,m,12
108, siva,20000,m,14
109, sivani,30000,f,15
110, mani,30000,m,12
111, manisha,300000,f,13
112, sivam,200000,m,12
113, varun,200000,m,13
```



In [10]:
emp = sc.textFile("/content/emp1.txt")
emp.count() # action

13

In [11]:
emp.getNumPartitions()

2

In [12]:
emp.collect() # action

['101, amar,90000,m,11',
 '102, amala,20000,f,12',
 '103, ankit,40000,m,13',
 '104, ankita,60000,f,13',
 '105, anusha,110000,f,12',
 '106, anuz,20000,m,11',
 '107, akash,100000,m,12',
 '108, siva,20000,m,14',
 '109, sivani,30000,f,15',
 '110, mani,30000,m,12',
 '111, manisha,300000,f,13',
 '112, sivam,200000,m,12',
 '113, varun,200000,m,13']

In [13]:
# task : for each dno, find maximum salary.
words = emp.map(lambda x : x.lower().split(','))
words.collect()


[['101', ' amar', '90000', 'm', '11'],
 ['102', ' amala', '20000', 'f', '12'],
 ['103', ' ankit', '40000', 'm', '13'],
 ['104', ' ankita', '60000', 'f', '13'],
 ['105', ' anusha', '110000', 'f', '12'],
 ['106', ' anuz', '20000', 'm', '11'],
 ['107', ' akash', '100000', 'm', '12'],
 ['108', ' siva', '20000', 'm', '14'],
 ['109', ' sivani', '30000', 'f', '15'],
 ['110', ' mani', '30000', 'm', '12'],
 ['111', ' manisha', '300000', 'f', '13'],
 ['112', ' sivam', '200000', 'm', '12'],
 ['113', ' varun', '200000', 'm', '13']]

In [17]:
dno_sal_pair = words.map(lambda x : (x[-1], int(x[2])))
dno_sal_pair.collect()

[('11', 90000),
 ('12', 20000),
 ('13', 40000),
 ('13', 60000),
 ('12', 110000),
 ('11', 20000),
 ('12', 100000),
 ('14', 20000),
 ('15', 30000),
 ('12', 30000),
 ('13', 300000),
 ('12', 200000),
 ('13', 200000)]

In [18]:
dnomax = dno_sal_pair.reduceByKey(lambda x, y : max(x, y))
dnomax.collect()

[('13', 300000), ('14', 20000), ('15', 30000), ('11', 90000), ('12', 200000)]

In [20]:
# for each dno, min salary.
dnomin = dno_sal_pair.reduceByKey(lambda x,y : min(x,y))
dnomin.collect()

[('13', 40000), ('14', 20000), ('15', 30000), ('11', 20000), ('12', 20000)]

In [21]:
# single grouping, multiple aggregations.
# note1: reduceByKey() cannot perform multiple aggregations.
# note2: reduceByKey() cannot perform non cumulative operations. eg. average
# solution: rdd.groupByKey()


In [25]:
# task : for each gender, find average salary
# sql : select gender, avg(salary) from emp group by gender;
gend_sal_pair = words.map(lambda x : (x[-2], int(x[2])))
gend_sal_pair.collect()

[('m', 90000),
 ('f', 20000),
 ('m', 40000),
 ('f', 60000),
 ('f', 110000),
 ('m', 20000),
 ('m', 100000),
 ('m', 20000),
 ('f', 30000),
 ('m', 30000),
 ('f', 300000),
 ('m', 200000),
 ('m', 200000)]

In [26]:
gend_grp = gend_sal_pair.groupByKey()
gend_grp.collect()

[('m', <pyspark.resultiterable.ResultIterable at 0x7feeee141790>),
 ('f', <pyspark.resultiterable.ResultIterable at 0x7feeee18d3d0>)]

In [27]:
gend_grp_list = gend_grp.mapValues(lambda x : list(x))
gend_grp_list.collect()

[('m', [90000, 40000, 20000, 100000, 20000, 30000, 200000, 200000]),
 ('f', [20000, 60000, 110000, 30000, 300000])]

In [28]:
gend_avgsal = gend_grp_list.mapValues(lambda x : round(sum(x)/len(x), 2))
gend_avgsal.collect()

[('m', 87500.0), ('f', 104000.0)]

In [29]:
# task : for each gender, find sum, count, avg, max, min
gend_grp_list.collect()

[('m', [90000, 40000, 20000, 100000, 20000, 30000, 200000, 200000]),
 ('f', [20000, 60000, 110000, 30000, 300000])]

In [37]:
# write a function to test a gender's sum, count, avg, max, min
def summary(x): # x is list
   tot = sum(x)
   cnt = len(x)
   avg = round(tot/cnt, 2)
   mx = max(x)
   mn = min(x)
   return[tot, cnt, avg, mx, mn]

In [38]:
# here take the females list ----> 'f', [20000, 60000, 110000, 30000, 300000])
summary([20000, 60000, 110000, 30000, 300000])

[520000, 5, 104000.0, 300000, 20000]

In [39]:
# gend_summary = gend_grp_list.mapValues(lambda x: summary(x))
gend_summary = gend_grp_list.mapValues(summary)
gend_summary.collect()

[('m', [700000, 8, 87500.0, 200000, 20000]),
 ('f', [520000, 5, 104000.0, 300000, 20000])]

In [41]:
# multi groupin with single aggregation
# task : for each dno and its sub group gender, find total salary.
dg_sal_pair = words.map(lambda x : ((x[-1], x[-2]), int(x[2])))
dg_sal_pair.collect()

[(('11', 'm'), 90000),
 (('12', 'f'), 20000),
 (('13', 'm'), 40000),
 (('13', 'f'), 60000),
 (('12', 'f'), 110000),
 (('11', 'm'), 20000),
 (('12', 'm'), 100000),
 (('14', 'm'), 20000),
 (('15', 'f'), 30000),
 (('12', 'm'), 30000),
 (('13', 'f'), 300000),
 (('12', 'm'), 200000),
 (('13', 'm'), 200000)]

In [42]:
dg_sal_tot = dg_sal_pair.reduceByKey(lambda x, y : x+y)
dg_sal_tot.collect()

[(('11', 'm'), 110000),
 (('12', 'f'), 130000),
 (('12', 'm'), 330000),
 (('13', 'm'), 240000),
 (('13', 'f'), 360000),
 (('14', 'm'), 20000),
 (('15', 'f'), 30000)]

In [45]:
# multi grouping with multiple aggregations.
# task: for each dno and sub group gender, find tot, count, avg, max, min
# sql : select dno, gend,, sum(sal), count(*), avg(sal), max(sal), min(sal) from emp group by dno, gend
dg_grp = dg_sal_pair.groupByKey()
dg_grp_list= dg_grp.mapValues(list)
dg_grp_list.collect()


[(('11', 'm'), [90000, 20000]),
 (('12', 'f'), [20000, 110000]),
 (('12', 'm'), [100000, 30000, 200000]),
 (('13', 'm'), [40000, 200000]),
 (('13', 'f'), [60000, 300000]),
 (('14', 'm'), [20000]),
 (('15', 'f'), [30000])]

In [47]:
dg_summary = dg_grp_list.mapValues(summary)
dg_summary.collect()

[(('11', 'm'), [110000, 2, 55000.0, 90000, 20000]),
 (('12', 'f'), [130000, 2, 65000.0, 110000, 20000]),
 (('12', 'm'), [330000, 3, 110000.0, 200000, 30000]),
 (('13', 'm'), [240000, 2, 120000.0, 200000, 40000]),
 (('13', 'f'), [360000, 2, 180000.0, 300000, 60000]),
 (('14', 'm'), [20000, 1, 20000.0, 20000, 20000]),
 (('15', 'f'), [30000, 1, 30000.0, 30000, 30000])]

In [48]:
dg_summary.saveAsTextFile('/content/summaryoutput')

In [55]:
def toLine(x):
  k = list(x[0])
  v = x[1]
  res = k + v
  res = ','.join([str(v) for v in res])
  return res
toLine((('15', 'f'), [30000, 1, 30000.0, 30000, 30000]))

'15,f,30000,1,30000.0,30000,30000'

In [56]:
resdata = dg_summary.map(toLine)
resdata.collect()

['11,m,110000,2,55000.0,90000,20000',
 '12,f,130000,2,65000.0,110000,20000',
 '12,m,330000,3,110000.0,200000,30000',
 '13,m,240000,2,120000.0,200000,40000',
 '13,f,360000,2,180000.0,300000,60000',
 '14,m,20000,1,20000.0,20000,20000',
 '15,f,30000,1,30000.0,30000,30000']

In [57]:
resdata.getNumPartitions()

2

In [59]:
resdata.coalesce(1).saveAsTextFile('/content/dgsummary')



```
# output file content:
11,m,110000,2,55000.0,90000,20000
12,f,130000,2,65000.0,110000,20000
12,m,330000,3,110000.0,200000,30000
13,m,240000,2,120000.0,200000,40000
13,f,360000,2,180000.0,300000,60000
14,m,20000,1,20000.0,20000,20000
15,f,30000,1,30000.0,30000,30000

```

